# OmniVoice fine-tune on Colab

Dataset: `pnnbao-ump/ngochuyen_voice`

Notebook này:
- mount Google Drive
- clone và cài OmniVoice
- tải dataset từ Hugging Face
- export thành `JSONL` đúng format OmniVoice
- tokenize audio
- fine-tune trên 1 GPU Colab


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Chọn cấu hình

Sửa các biến dưới đây nếu cần.
- `BASE_DIR`: nơi lưu dữ liệu, token, checkpoint trên Drive
- `HF_DATASET`: dataset nguồn
- `TRAIN_STEPS`: số bước train
- `BATCH_TOKENS`: giảm xuống nếu bị OOM


In [ ]:
BASE_DIR = '/content/drive/MyDrive/omnivoice_ngochuyen'
HF_DATASET = 'pnnbao-ump/ngochuyen_voice'
TRAIN_STEPS = 3000
BATCH_TOKENS = 512
DEV_RATIO = 0.02
SEED = 42


## 3. Clone repo và cài dependencies

In [ ]:
%cd /content
!git clone https://github.com/k2-fsa/OmniVoice.git
%cd /content/OmniVoice
!pip install -U pip setuptools wheel
!pip install -e .
!pip install accelerate tensorboard datasets soundfile

## 4. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 5. Tải dataset và chia train/dev

In [ ]:
from datasets import load_dataset

ds = load_dataset(HF_DATASET, split='train')
splits = ds.train_test_split(test_size=DEV_RATIO, seed=SEED)
train_ds = splits['train']
dev_ds = splits['test']

print(train_ds)
print(dev_ds)
print(train_ds[0])

## 6. Export WAV + JSONL đúng format OmniVoice

Format mỗi dòng:
```json
{"id": "train_00001", "audio_path": "/abs/path.wav", "text": "xin chao", "language_id": "vi"}
```

In [ ]:
import os
import json
import soundfile as sf

train_wav_dir = f'{BASE_DIR}/wavs/train'
dev_wav_dir = f'{BASE_DIR}/wavs/dev'
os.makedirs(train_wav_dir, exist_ok=True)
os.makedirs(dev_wav_dir, exist_ok=True)

def clean_text(text):
    text = (text or '').strip()
    text = ' '.join(text.split())
    return text

def export_jsonl(ds, wav_dir, jsonl_path, prefix):
    kept = 0
    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for i, row in enumerate(ds):
            text = clean_text(row.get('transcription', ''))
            if not text:
                continue

            audio = row['audio']
            wav_path = os.path.join(wav_dir, f'{prefix}_{kept:05d}.wav')
            sf.write(wav_path, audio['array'], audio['sampling_rate'])

            item = {
                'id': f'{prefix}_{kept:05d}',
                'audio_path': wav_path,
                'text': text,
                'language_id': 'vi'
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            kept += 1
    return kept

train_count = export_jsonl(train_ds, train_wav_dir, f'{BASE_DIR}/train.jsonl', 'train')
dev_count = export_jsonl(dev_ds, dev_wav_dir, f'{BASE_DIR}/dev.jsonl', 'dev')

print('train items:', train_count)
print('dev items:', dev_count)

## 7. Tạo config cho fine-tune

In [ ]:
import json
import os

gpu_name = ''
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0).upper()
except Exception:
    pass

mixed_precision = 'bf16' if ('A100' in gpu_name or 'H100' in gpu_name) else 'fp16'

train_config = {
    'llm_name_or_path': 'Qwen/Qwen3-0.6B',
    'audio_vocab_size': 1025,
    'audio_mask_id': 1024,
    'num_audio_codebook': 8,
    'audio_codebook_weights': [8, 8, 6, 6, 4, 4, 2, 2],
    'drop_cond_ratio': 0.1,
    'prompt_ratio_range': [0.0, 0.3],
    'mask_ratio_range': [0.0, 1.0],
    'language_ratio': 1.0,
    'use_pinyin_ratio': 0.0,
    'instruct_ratio': 0.0,
    'only_instruct_ratio': 0.0,
    'resume_from_checkpoint': None,
    'init_from_checkpoint': 'k2-fsa/OmniVoice',
    'learning_rate': 5e-5,
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'steps': TRAIN_STEPS,
    'seed': SEED,
    'warmup_type': 'ratio',
    'warmup_ratio': 0.01,
    'warmup_steps': 0,
    'batch_tokens': BATCH_TOKENS,
    'gradient_accumulation_steps': 1,
    'num_workers': 2,
    'mixed_precision': mixed_precision,
    'allow_tf32': True,
    'logging_steps': 50,
    'eval_steps': 500,
    'save_steps': 500,
    'keep_last_n_checkpoints': -1
}

data_config = {
    'train': [{'manifest_path': [f'{BASE_DIR}/tokens/train/data.lst']}],
    'dev': [{'manifest_path': [f'{BASE_DIR}/tokens/dev/data.lst']}]
}

os.makedirs(BASE_DIR, exist_ok=True)
with open(f'{BASE_DIR}/train_config.json', 'w', encoding='utf-8') as f:
    json.dump(train_config, f, ensure_ascii=False, indent=2)
with open(f'{BASE_DIR}/data_config.json', 'w', encoding='utf-8') as f:
    json.dump(data_config, f, ensure_ascii=False, indent=2)

print('mixed_precision =', mixed_precision)
print('saved config files')

## 8. Tokenize train split

In [ ]:
%cd /content/OmniVoice
!CUDA_VISIBLE_DEVICES=0 python -m omnivoice.scripts.extract_audio_tokens \
  --input_jsonl /content/drive/MyDrive/omnivoice_ngochuyen/train.jsonl \
  --tar_output_pattern /content/drive/MyDrive/omnivoice_ngochuyen/tokens/train/audios/shard-%06d.tar \
  --jsonl_output_pattern /content/drive/MyDrive/omnivoice_ngochuyen/tokens/train/txts/shard-%06d.jsonl \
  --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
  --nj_per_gpu 2 \
  --shuffle True

## 9. Tokenize dev split

In [ ]:
%cd /content/OmniVoice
!CUDA_VISIBLE_DEVICES=0 python -m omnivoice.scripts.extract_audio_tokens \
  --input_jsonl /content/drive/MyDrive/omnivoice_ngochuyen/dev.jsonl \
  --tar_output_pattern /content/drive/MyDrive/omnivoice_ngochuyen/tokens/dev/audios/shard-%06d.tar \
  --jsonl_output_pattern /content/drive/MyDrive/omnivoice_ngochuyen/tokens/dev/txts/shard-%06d.jsonl \
  --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
  --nj_per_gpu 2 \
  --shuffle True

## 10. Train

In [ ]:
%cd /content/OmniVoice
!accelerate launch \
  --gpu_ids '0' \
  --num_processes 1 \
  -m omnivoice.cli.train \
  --train_config /content/drive/MyDrive/omnivoice_ngochuyen/train_config.json \
  --data_config /content/drive/MyDrive/omnivoice_ngochuyen/data_config.json \
  --output_dir /content/drive/MyDrive/omnivoice_ngochuyen/exp

## 11. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/omnivoice_ngochuyen/exp

## 12. Ghi chú

- Nếu bị `CUDA out of memory`, giảm `BATCH_TOKENS` từ `512` xuống `256`.
- Nếu runtime bị reset, chỉ cần mount Drive lại và chạy lại từ cell cài đặt hoặc cell train tùy phần đã hoàn thành.
- Dataset `pnnbao-ump/ngochuyen_voice` có license `cc-by-nc-4.0`, không phù hợp cho mục đích thương mại.
